# Evaluating attribution faithfulness with `autolrp.eval`

An attribution map is only useful if it actually identifies the features the model uses. The `autolrp.eval` module ships four model-agnostic, modality-agnostic metrics that measure exactly that — they take a `(model, x, R)` tuple and tell you whether `R` is faithful, not just pretty.

| function | what it measures | reference |
|---|---|---|
| `perturbation_curve` | model-score curve as top-\|R\| positions are removed (deletion) or added back (insertion); AUC | Petsiuk et al. 2018 (RISE) |
| `aopc` | single-scalar mean drop in score across the perturbation curve | Bach 2015 / Samek 2017 |
| `sanity_check_cascade` | randomize model layers top-to-bottom; faithful attribution decays sharply, model-independent attribution stays constant | Adebayo et al. 2018 |
| `sensitivity_correlation` | Pearson correlation between summed `R` on random subsets and the model's actual output drop when those subsets are masked | Ancona 2018 / Bhatt 2021 |

This notebook runs all four against **three different attribution maps** for the same `(model, input)` to show the metrics actually distinguish:
- **LRP-composite**: z⁺ on conv, ε elsewhere — a real, model-dependent attribution.
- **Sobel edge filter**: a fixed image filter; doesn't see the model at all. The classic Adebayo 2018 sanity-check failure case.
- **Uniform random**: noise; a lower bound.

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

import autolrp
from autolrp import LRPConfig, eval as alrp_eval
import _common  # noqa  — applies shared rcParams

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)

In [ ]:
# Real model on a real image — ResNet-18 ImageNet weights.
# Resize to 128×128 so per-step forwards stay fast on CPU.
model = resnet18(weights=ResNet18_Weights.DEFAULT).eval().to(device)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
preprocess = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
x = preprocess(Image.open('../../../data/cat0.jpg').convert('RGB')).unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(x)
target = int(logits.argmax(-1).item())
with open('../../../data/imagenet_classes.txt') as f:
    classes = [c.strip() for c in f if c.strip()]
print(f'predicted: class {target} = {classes[target]}')

## The three attribution methods being compared

We compute `R_lrp` via autoLRP-composite (a faithful method), `R_sobel` via a Sobel edge filter (a model-independent baseline — it doesn't depend on the model's weights at all, and the Adebayo 2018 paper's classic failure case for many "saliency" methods), and `R_random` as Gaussian noise (a lower bound). All three are reduced to per-pixel granularity by summing across the channel dim — eval metrics broadcast them back to `(1, 3, H, W)` automatically.

In [ ]:
# 1. LRP attribution.
xt = autolrp.tensor(x.clone())
model(xt)[0, target].lrp(config=LRPConfig.composite())
R_lrp = xt.relevance.detach().sum(dim=1, keepdim=True)        # (1,1,H,W)

# 2. Sobel edge filter — model-independent.
gray = x.mean(dim=1, keepdim=True)                             # (1,1,H,W)
sx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                   dtype=x.dtype, device=device).view(1, 1, 3, 3)
sy = sx.transpose(2, 3)
gx = F.conv2d(gray, sx, padding=1)
gy = F.conv2d(gray, sy, padding=1)
R_sobel = (gx**2 + gy**2).sqrt()                               # (1,1,H,W)

# 3. Uniform random — lower bound.
R_random = torch.randn(1, 1, 128, 128, generator=torch.Generator(device=device).manual_seed(42),
                        device=device)

METHODS = {'LRP-composite': R_lrp, 'Sobel-edge': R_sobel, 'random': R_random}
for name, R in METHODS.items():
    print(f'{name:18s}  shape={tuple(R.shape)}  '
          f'min={R.min().item():+.3f}  max={R.max().item():+.3f}')

## Visualize the three maps for orientation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
img = (x[0].cpu() * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1))
img = img.clamp(0, 1).permute(1, 2, 0).numpy()
axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')
for ax, (name, R) in zip(axes[1:], METHODS.items()):
    m = R[0, 0].detach().cpu().numpy()
    v = abs(m).max() + 1e-9
    ax.imshow(m, cmap='coolwarm', vmin=-v, vmax=v)
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

## Metric 1 — perturbation curves (Petsiuk 2018 / RISE)

Replace the top-k% positions by-\|R\| with the input's per-channel mean and watch the predicted-class probability collapse. Faithful attribution: sharp drop in **deletion** (we're removing the positions the model actually uses) and sharp rise in **insertion** (we're adding them back). Unfaithful methods produce flat / diagonal curves regardless of which positions are removed.

In [ ]:
results = {name: {} for name in METHODS}
for name, R in METHODS.items():
    for mode in ('deletion', 'insertion'):
        out = alrp_eval.perturbation_curve(
            model, x, R, mode=mode, n_steps=20, baseline='mean')
        results[name][mode] = out
    print(f'{name:18s}  '
          f'deletion AUC = {results[name]["deletion"]["auc"]:.3f}  '
          f'insertion AUC = {results[name]["insertion"]["auc"]:.3f}')

In [ ]:
# Plot both curves.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {'LRP-composite': '#cc3333', 'Sobel-edge': '#3b6fb6', 'random': '#888888'}
for ax, mode, title in [
    (axes[0], 'deletion',  'Deletion: remove top-|R| → lower is better'),
    (axes[1], 'insertion', 'Insertion: add top-|R| → higher is better'),
]:
    for name in METHODS:
        c = results[name][mode]
        ax.plot(c['fractions'], c['scores'], '-o', label=name,
                 color=colors[name], markersize=3)
    ax.set_xlabel('fraction of input perturbed')
    ax.set_ylabel('predicted-class probability')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9, frameon=False)
plt.tight_layout(); plt.show()

## Metric 2 — AOPC (Bach 2015 / Samek 2017)

One-number summary: the mean drop in predicted probability across the perturbation curve. `mode='deletion'` removes the most-relevant positions first (MoRF in Samek et al.); `mode='insertion'` the least-relevant first (LeRF). Faithful attribution: **MoRF AOPC large** (removing top positions causes large drops); **LeRF AOPC small** (removing bottom positions barely affects the score).

In [ ]:
rows = []
for name, R in METHODS.items():
    morf = alrp_eval.aopc(model, x, R, mode='deletion',  n_steps=20, baseline='mean')   # most relevant first
    lerf = alrp_eval.aopc(model, x, R, mode='insertion', n_steps=20, baseline='mean')   # least relevant first
    rows.append((name, morf, lerf, morf - lerf))
print(f'{"method":18s}  MoRF AOPC   LeRF AOPC   Δ (MoRF − LeRF, ↑ better)')
for name, morf, lerf, delta in rows:
    print(f'{name:18s}  {morf:+.4f}    {lerf:+.4f}    {delta:+.4f}')

# Bar chart.
names = [r[0] for r in rows]
morf_vals = [r[1] for r in rows]; lerf_vals = [r[2] for r in rows]
fig, ax = plt.subplots(figsize=(7, 3.8))
xpos = np.arange(len(names))
ax.bar(xpos - 0.18, morf_vals, width=0.36, color='#cc3333', label='MoRF (↑ better)')
ax.bar(xpos + 0.18, lerf_vals, width=0.36, color='#3b6fb6', label='LeRF (↓ better)')
ax.set_xticks(xpos); ax.set_xticklabels(names)
ax.set_ylabel('AOPC')
ax.axhline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=9, frameon=False)
plt.tight_layout(); plt.show()

## Metric 3 — sanity-check cascade (Adebayo 2018)

Randomize the model's layers top-to-bottom and recompute the attribution after each randomization. A **faithful** method must depend on the model's weights — randomizing weights should produce a *different* attribution, so cosine similarity to the original drops as more layers are randomized. A method that doesn't depend on the model (Sobel: it's a fixed filter) produces *the same* attribution regardless of weights — its similarity stays at 1. This is the classic Adebayo falsification.

In [ ]:
# The cascade needs an `attribute_fn(model, x) -> R` callable.
# We supply one per attribution method.

def attr_lrp(m, x_):
    xt = autolrp.tensor(x_.clone())
    m(xt)[0, target].lrp(config=LRPConfig.composite())
    return xt.relevance.detach().sum(dim=1, keepdim=True)

def attr_sobel(m, x_):
    g = x_.mean(dim=1, keepdim=True)
    sx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=x_.dtype, device=x_.device).view(1,1,3,3)
    sy = sx.transpose(2,3)
    return (F.conv2d(g, sx, padding=1)**2 + F.conv2d(g, sy, padding=1)**2).sqrt()

def attr_random(m, x_):
    # Seed kept fixed so 'random' isn't being re-sampled — the
    # cascade should still see *the same* random map regardless
    # of layer randomization, mimicking Sobel's model-independence.
    return torch.randn(1, 1, *x_.shape[-2:],
                        generator=torch.Generator(device=x_.device).manual_seed(42),
                        device=x_.device)

ATTR_FNS = {'LRP-composite': attr_lrp, 'Sobel-edge': attr_sobel, 'random': attr_random}

cascade = {}
for name, fn in ATTR_FNS.items():
    out = alrp_eval.sanity_check_cascade(model, x, fn, similarity='cosine')
    cascade[name] = out
    print(f'{name:18s}  '
          f'first={out["similarities"][0]:+.3f}  '
          f'last={out["similarities"][-1]:+.3f}  '
          f'  Δ={out["similarities"][-1] - out["similarities"][0]:+.3f}')

In [ ]:
# Plot the similarity decay curves.
fig, ax = plt.subplots(figsize=(9, 4))
for name, c in cascade.items():
    n = len(c['similarities'])
    ax.plot(range(n), c['similarities'], '-o', label=name,
             color=colors[name], markersize=3)
ax.axhline(1, color='gray', linewidth=0.4, linestyle='--')
ax.set_xlabel('layers randomized (cumulative, top → bottom)')
ax.set_ylabel('cosine similarity to original R')
ax.set_title('Cascade randomization: faithful attribution decays, '
              'model-independent stays flat', fontsize=10)
ax.legend(fontsize=9, frameon=False)
ax.set_ylim(-0.1, 1.05)
plt.tight_layout(); plt.show()

## Metric 4 — sensitivity correlation (Ancona 2018 / Bhatt 2021)

For 100 random subsets of input positions, correlate the attribution's claim (Σ `R` over the subset) with the model's actual response (output drop when that subset is masked). Faithful attribution: claims and actual responses correlate highly. Unfaithful: low or zero correlation.

This is the only metric here that uses the **signed** attribution values (the others rank by \|R\|). For unsigned methods like Sobel (always non-negative) and random (zero-mean), the signal is naturally weaker.

In [ ]:
for name, R in METHODS.items():
    out = alrp_eval.sensitivity_correlation(
        model, x, R, subset_size=0.2, n_samples=100,
        baseline='mean', seed=0)
    print(f'{name:18s}  Pearson correlation = {out["correlation"]:+.3f}  '
          f'(higher = more faithful)')

## Reading the four metrics together

Each metric tests a slightly different facet of faithfulness, and the metrics don't always agree on which method is best. That disagreement is the *useful* signal of running all four:

- **Perturbation curves and AOPC** test whether the *ranking* of `|R|` matches what the model uses. On natural images, Sobel (which highlights edges) often scores comparably to faithful methods on these metrics — not because Sobel is faithful, but because *edges happen to be where ImageNet models look*. Methods correlated with task-relevant features by accident pass perturbation tests.
- **Sensitivity correlation** has the same vulnerability: it measures whether the *signed* `R` predicts the model's behavioral response. Sobel can score high here for the same reason — its values correlate with information-rich pixels.
- **The sanity-check cascade is the only metric that directly falsifies model-independence.** Sobel and random stay at cosine = 1 throughout the cascade (their `R` doesn't depend on the model's weights at all). LRP-composite decays sharply. This is the classic Adebayo result and the most important falsification test in the suite.

The takeaway: **no single metric is sufficient**. A method that scores well on perturbation curves but fails the cascade is a model-independent feature detector, not a faithful attribution. A method that passes all four is what the recent attribution literature calls "empirically faithful."

### Plug in your own attribution

All four functions accept any `R` of the right shape:

```python
# Captum's Integrated Gradients, etc., return tensors of the same
# shape as x — drop them straight in:
R_ig = captum_method.attribute(x, target=target).detach()
alrp_eval.aopc(model, x, R_ig)                  # → float
alrp_eval.perturbation_curve(model, x, R_ig)    # → dict with curves
```

For a much wider metric library covering ~30 more variants (Infidelity, Max-Sensitivity, IROF, sparsity, localization, …), see [Quantus](https://github.com/understandable-machine-intelligence-lab/Quantus). Its `(model, x_batch, a_batch)` API is identical in shape to ours so the same `R` plugs straight in.

### References
- Bach, Binder, Montavon, Klauschen, Müller, Samek (2015). *On Pixel-Wise Explanations for Non-Linear Classifier Decisions by Layer-Wise Relevance Propagation*. PLoS ONE.
- Samek, Binder, Montavon, Lapuschkin, Müller (2017). *Evaluating the Visualization of What a Deep Neural Network Has Learned*. IEEE TNNLS.
- Petsiuk, Das, Saenko (2018). *RISE: Randomized Input Sampling for Explanation of Black-box Models*. BMVC.
- Adebayo, Gilmer, Muelly, Goodfellow, Hardt, Kim (2018). *Sanity Checks for Saliency Maps*. NeurIPS.
- Ancona, Ceolini, Öztireli, Gross (2018). *Towards Better Understanding of Gradient-based Attribution Methods*. ICLR.
- Bhatt, Weller, Moura (2021). *Evaluating and Aggregating Feature-based Model Explanations*. IJCAI.